# Week 2 · Tutorial 3 — Correlation and mutual information

**Covers videos 2.6 – 2.7:** correlation (one number for a relationship, and what it drops) · mutual information (dependence without assuming a shape).

**Datasets**
* `microgrid_hourly.csv` — hourly data from a small island microgrid for 2023: `load_kw`, `temperature_c`, `hour`, `weekend`,
  `pv_kw` (solar output), `wind_ms`, and `random_signal` (pure noise, for comparison). **One row is one hour.**
* `monthly_region.csv` — monthly fan sales (`fan_sales_units`), electricity use (`electricity_gwh`) and mean temperature for a region, 2010–2022. **One row is one month.**

As always, we explore **only the training part** (the first 70 % of the hours).

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

In [ ]:
grid = pd.read_csv("data/microgrid_hourly.csv", parse_dates=["timestamp"]).sort_values("timestamp")
train = grid.iloc[:int(0.7 * len(grid))].copy()
print("training hours:", len(train))
train.head()

## 1. The correlation matrix

The Pearson coefficient *r* measures how well a **straight line** describes the relationship between two variables.
It lies between −1 and 1 and has a sign. The Spearman coefficient is the same calculation on **ranks** — it detects any
*monotonic* relationship.

In [ ]:
cols = ["load_kw", "temperature_c", "hour", "weekend", "pv_kw", "wind_ms", "random_signal"]
cm = train[cols].corr()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=60, ha="right")
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
plt.colorbar(im); plt.show()

### ✏️ Your turn 1

Compute the correlation of every column with `load_kw`, once with Pearson and once with Spearman.

Change `method` to `"spearman"` (the starter uses Pearson) and run the cell again; write down both columns of numbers.
The cell prints the result for the chosen method, and the number of variables with an absolute correlation above 0.15.

In [ ]:
method = "pearson"        # TODO: also run with "spearman"

corr_with_load = train[cols].corr(method=method)["load_kw"].drop("load_kw")
print(method)
print(corr_with_load.round(3))
print("variables with |r| > 0.15:", int((corr_with_load.abs() > 0.15).sum()))

## 2. A strong relationship with correlation near zero

PV output depends on the hour of the day in a very regular way — a bell curve around noon. But that is not a straight line.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(train["hour"], train["pv_kw"], s=4, alpha=0.3)
ax.set_xlabel("hour of day"); ax.set_ylabel("PV output (kW)")
plt.show()
print("Pearson r(hour, pv_kw) =", round(train["hour"].corr(train["pv_kw"]), 3))

### ✏️ Your turn 2

Change the shape of the question so that it *is* a straight line.

Create the feature `dist_noon = |hour − 12|` (the distance from midday in hours, `(train["hour"] - 12).abs()`) and compute
its Pearson correlation with `pv_kw` (3 decimals), and its Spearman correlation too.

**What to change** — edit the cell exactly like this, then run it:

Change `train["dist_noon"] = 0.0` into `train["dist_noon"] = (train["hour"] - 12).abs()`

In [ ]:
train["dist_noon"] = 0.0      # TODO: (train["hour"] - 12).abs()

r_hour = train["hour"].corr(train["pv_kw"])
r_dist = train["dist_noon"].corr(train["pv_kw"])
rho_dist = train["dist_noon"].corr(train["pv_kw"], method="spearman")
print("Pearson  r(hour, pv)        :", round(r_hour, 3))
print("Pearson  r(dist_noon, pv)   :", round(r_dist, 3))
print("Spearman rho(dist_noon, pv) :", round(rho_dist, 3))

## 3. Why do two things move together? — a confounder

`monthly_region.csv`: fan sales and electricity use are strongly correlated. Do fans cause electricity use? Both follow the
**temperature** (hot months: fans sold, air conditioning running). A *common cause* creates correlation without any direct effect.

In [ ]:
monthly = pd.read_csv("data/monthly_region.csv")
print(monthly.shape)
print(monthly[["temperature_c", "fan_sales_units", "electricity_gwh"]].corr().round(3))

### ✏️ Your turn 3

Remove the part of each variable that is explained by temperature, and correlate what is left.

The starter uses only **two** temperature bins. Replace the bin edges by `[-10, 0, 4, 8, 12, 16, 20, 40]` (seven bins). The
cell subtracts the **bin mean** from `fan_sales_units` and `electricity_gwh` and computes the correlation of the remainders
(the *within-temperature-bin* correlation).

**What to change** — edit the cell exactly like this, then run it:

Change `edges = [-10, 8, 40]` into `edges = [-10, 0, 4, 8, 12, 16, 20, 40]`

In [ ]:
edges = [-10, 8, 40]      # TODO: [-10, 0, 4, 8, 12, 16, 20, 40]

monthly["temp_bin"] = pd.cut(monthly["temperature_c"], edges)
for c in ["fan_sales_units", "electricity_gwh"]:
    monthly[c + "_resid"] = monthly[c] - monthly.groupby("temp_bin", observed=True)[c].transform("mean")

overall = monthly["fan_sales_units"].corr(monthly["electricity_gwh"])
within = monthly["fan_sales_units_resid"].corr(monthly["electricity_gwh_resid"])
print("number of bins used:", monthly["temp_bin"].nunique())
print("overall correlation fans vs electricity :", round(overall, 3))
print("within-temperature-bin correlation       :", round(within, 3))

## 4. Mutual information

Mutual information (MI) asks: *how much less uncertain is the target once I know this variable?* — without assuming any
shape. MI is 0 for independent variables, has **no sign**, and is **estimated** from the data (here with k-nearest neighbours),
so its value depends on settings and a random seed.

### ✏️ Your turn 4

Compute the MI between each candidate input and `load_kw` on the training set.

1. Set `n_neighbors = 10` (starter: 3) in the `TODO` line.
2. The cell prints the MI of each feature (3 decimals) in descending order, and the feature with the highest MI.

**What to change** — edit the cell exactly like this, then run it:

Change `n_neighbors = 3` into `n_neighbors = 10`

In [ ]:
from sklearn.feature_selection import mutual_info_regression

features = ["temperature_c", "hour", "weekend", "pv_kw", "wind_ms", "random_signal"]
n_neighbors = 3        # TODO: 10

mi = pd.Series(
    mutual_info_regression(train[features], train["load_kw"], n_neighbors=n_neighbors, random_state=0),
    index=features).sort_values(ascending=False)
print("n_neighbors =", n_neighbors)
print(mi.round(3))
print("highest MI:", mi.idxmax())

### ✏️ Your turn 5

The two measures can disagree. Compute, for the pair **(`hour` → `pv_kw`)**:
- the Pearson correlation `r_pv` (3 decimals)
- the MI `mi_pv` (use `mutual_info_regression(train[["hour"]], train["pv_kw"], random_state=0)`; 3 decimals)

and the same MI for the noise column: `mi_noise` = MI of `random_signal` with `pv_kw`.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
r_pv = np.nan
mi_pv = np.nan
mi_noise = np.nan
```
by
```python
r_pv = train["hour"].corr(train["pv_kw"])
mi_pv = mutual_info_regression(train[["hour"]], train["pv_kw"], random_state=0)[0]
mi_noise = mutual_info_regression(train[["random_signal"]], train["pv_kw"], random_state=0)[0]
```

In [ ]:
r_pv = np.nan         # TODO
mi_pv = np.nan        # TODO
mi_noise = np.nan     # TODO

print("Pearson r(hour, pv) :", round(r_pv, 3))
print("MI(hour; pv)        :", round(mi_pv, 3))
print("MI(noise; pv)       :", round(mi_noise, 3))

### ✏️ Your turn 6

MI is **estimated**. Repeat the estimate of MI(`temperature_c` → `load_kw`) with five different random seeds
(`random_state` = 0, 1, 2, 3, 4) with `n_neighbors = 10` and store them in the list `mi_values`. The cell prints the smallest, the largest,
and the difference (spread).

**What to change** — replace the line `mi_values = []` by this one (a *list comprehension*: it repeats the estimate for every seed `s` from 0 to 4):

```python
mi_values = [mutual_info_regression(train[["temperature_c"]], train["load_kw"], n_neighbors=10, random_state=s)[0] for s in range(5)]
```

In [ ]:
mi_values = []     # TODO: one MI value per seed

if mi_values:
    print("seeds:", len(mi_values))
    print("min / max:", round(min(mi_values), 4), round(max(mi_values), 4))
    print("spread:", round(max(mi_values) - min(mi_values), 4))

## Summary

* Correlation summarises a **straight line**. A strong but non-linear relationship (PV vs. hour) can have *r* ≈ 0 — transform the variable, or use a different measure.
* Two variables can move together because of a **common cause**; control for it before interpreting.
* Mutual information sees any shape but has no sign, no natural scale and is *estimated*. Use both — and then draw the figure.